In [2]:
import json
import zipfile
import tempfile
import os
import shutil
import numpy as np
import keras
import torch

In [3]:
@keras.saving.register_keras_serializable()
class LocalResponseNormalization(keras.layers.Layer):
    def call(self, x):
        return torch.nn.functional.local_response_norm(x, size=5, alpha=1e-4, beta=0.75, k=2.0)

    def compute_output_shape(self, input_shape):
        return input_shape

In [4]:
model = keras.models.load_model("model/alexnet.keras", compile=False,
                                custom_objects={"LocalResponseNormalization": LocalResponseNormalization})

In [5]:
with open("model/alexnet_architecture.json", "w") as f:
    f.write(model.to_json())

In [6]:
weight_dict = {}
metadata = {}

In [7]:
for i, layer in enumerate(model.layers):
    weights = layer.get_weights()
    if weights:
        for j, w in enumerate(weights):
            # Per-tensor asymmetric quantization
            w_min = w.min()
            w_max = w.max()

            # Handle constant tensors (e.g., biases that are all zero)
            if w_max == w_min:
                scale = 1.0
                zero_point = 0
                int8_w = np.zeros_like(w, dtype=np.int8)
            else:
                # Symmetric INT8: range [-127, 127]
                scale = max(abs(w_min), abs(w_max)) / 127.0
                if scale == 0:
                    scale = 1.0
                int8_w = np.round(w / scale).clip(-127, 127).astype(np.int8)
                zero_point = 0  # symmetric

            weight_dict[f"layer_{i}_weight_{j}"] = int8_w
            metadata[f"layer_{i}_weight_{j}"] = {
                "scale": float(scale),
                "zero_point": int(zero_point),
                "shape": list(w.shape),
                "dtype": str(w.dtype)
            }

In [8]:
np.savez_compressed("model/alexnet_int8_weights.npz", **weight_dict)
with open("model/alexnet_int8_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [9]:
npz_size = os.path.getsize("model/alexnet_int8_weights.npz") / 1024 / 1024
print(f"INT8 weights npz: {npz_size:.2f} MB")

INT8 weights npz: 40.71 MB


In [10]:
with zipfile.ZipFile("model/alexnet_int8_weights.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("model/alexnet_int8_weights.npz", arcname="alexnet_int8_weights.npz")
    z.write("model/alexnet_int8_metadata.json", arcname="alexnet_int8_metadata.json")

zip_size = os.path.getsize("model/alexnet_int8_weights.zip") / 1024 / 1024
print(f"Zipped: {zip_size:.2f} MB")

Zipped: 40.73 MB


In [11]:
def load_int8_model(arch_path, zip_path):
    with open(arch_path, "r") as f:
        model = keras.models.model_from_json(f.read(), custom_objects={"LocalResponseNormalization": LocalResponseNormalization})

    temp_dir = tempfile.mkdtemp()
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(temp_dir)

        npz_path = os.path.join(temp_dir, "alexnet_int8_weights.npz")
        meta_path = os.path.join(temp_dir, "alexnet_int8_metadata.json")

        with open(meta_path, "r") as f:
            metadata = json.load(f)

        with np.load(npz_path) as data:
            weight_arrays = {k: data[k].copy() for k in data.files}

        for i, layer in enumerate(model.layers):
            original_weights = layer.get_weights()
            if not original_weights:
                continue
            new_weights = []
            for j in range(len(original_weights)):
                key = f"layer_{i}_weight_{j}"
                int8_w = weight_arrays[key]
                scale = metadata[key]["scale"]
                float_w = int8_w.astype(np.float32) * scale
                new_weights.append(float_w)
            layer.set_weights(new_weights)

    finally:
        try:
            shutil.rmtree(temp_dir, ignore_errors=True)
        except Exception as e:
            print(f"Warning: could not clean temp dir {temp_dir}: {e}")
    return model

In [12]:
model = load_int8_model("model/alexnet_architecture.json", "model/alexnet_int8_weights.zip")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 3, 224, 224)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d (ZeroPadding2D)  │ (None, 3, 228, 228)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 96, 55, 55)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lrn1                            │ (None, 96, 55, 55)     │             0 │
│ (LocalResponseNormalization)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 96, 27, 27)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 256, 27, 27)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lrn2                            │ (None, 256, 27, 27)    │             0 │
│ (LocalResponseNormalization)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 256, 13, 13)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 384, 13, 13)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 384, 13, 13)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5 (Conv2D)                  │ (None, 256, 13, 13)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool5 (MaxPooling2D)            │ (None, 256, 6, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense3 (Dense)                  │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,378,344 (237.95 MB)

 Trainable params: 62,378,344 (237.95 MB)

 Non-trainable params: 0 (0.00 B)